# Pipeline d'Inférence en Production - Prédictions 3 Jours

Ce notebook exécute le pipeline complet d'inférence:
1. Charge du modèle depuis MLflow en Production
2. Génération des données pour 3 jours
3. Exécution des prédictions
4. Stockage en base de données PostgreSQL
5. Vérification des résultats

## Configuration

Définir les paramètres pour l'inférence

In [6]:
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(".env"), override=True)      # config
load_dotenv(find_dotenv(".env.secrets"), override=True)   # secrets only useful on local environment


True

In [7]:
# Paramètres MLflow


MLFLOW_TRACKING_URI = "https://jenedai-mlflow.hf.space/"  # À adapter selon votre déploiement
EXPERIMENT_NAME = "enedis_project"
MODEL_NAME = "enedis_model"  # Nom du modèle à charger depuis Production

# Construire l'URI PostgreSQL
DB_URI = os.environ.get("NEON_POSTGRES_URI", "postgresql://localhost:5432/enedis_db")
ProjectName = os.environ.get("PROJECT_NAME", "enedis_project")

# Paramètres d'inférence
N_DAYS = 3  # Nombre de jours à prédire
N_SAMPLES_PER_DAY = 24  # Hourly

# Features attendues par le modèle
FEATURE_COLUMNS = [
    'temperature', 
    'humidity', 
    'wind_speed', 
    'cloud_cover',
    'day_of_week', 
    'hour', 
    'holiday'
]

print(f"✓ Configuration chargée")
print(f"  MLflow: {MLFLOW_TRACKING_URI}")
print(f"  Modèle: {MODEL_NAME}")
print(f"  Prédictions: {N_DAYS} jours x {N_SAMPLES_PER_DAY} échantillons/jour")

✓ Configuration chargée
  MLflow: https://jenedai-mlflow.hf.space/
  Modèle: enedis_model
  Prédictions: 3 jours x 24 échantillons/jour


## Initialisation du Pipeline

In [8]:
from utils.prediction_pipeline import PredictionPipeline
import logging

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Créer le pipeline
pipeline = PredictionPipeline(
    mlflow_uri=MLFLOW_TRACKING_URI,
    experiment_name=EXPERIMENT_NAME,
    db_uri=DB_URI
)

print("✓ Pipeline d'inférence créé")

INFO:utils.prediction_pipeline:Pipeline d'inférence initialisé


✓ Pipeline d'inférence créé


## Export des Résultats (Optionnel)

In [9]:
# Récupérer les prédictions générées en mémoire
predictions = pipeline.get_predictions_df()

if predictions is not None:
    print(f"Prédictions en mémoire: {len(predictions)} lignes x {len(predictions.columns)} colonnes")
    print("\nColonnes:")
    print(predictions.columns.tolist())
    
    # Afficher un aperçu
    print("\nAperçu des données:")
    print(predictions.head().to_string())
else:
    print("Pas de prédictions disponibles")

Pas de prédictions disponibles


## Récupération des Prédictions Générées

In [10]:
# Exécuter le pipeline complet
success, predictions_df = pipeline.run_full_pipeline(
    model_name=MODEL_NAME,
    feature_columns=FEATURE_COLUMNS,
    n_days=N_DAYS,
    n_samples_per_day=N_SAMPLES_PER_DAY
)

if success:
    print("\n✓ Pipeline exécuté avec succès!\n")
    
    if predictions_df is not None:
        print(f"Nombre de prédictions stockées: {len(predictions_df)}")
        print("\nDernières prédictions:")
        print(predictions_df.head(10).to_string())
        
        # Statistiques des prédictions
        print("\n--- Statistiques des prédictions ---")
        if 'prediction' in predictions_df.columns:
            print(f"Min prédiction: {predictions_df['prediction'].min():.4f}")
            print(f"Max prédiction: {predictions_df['prediction'].max():.4f}")
            print(f"Moyenne: {predictions_df['prediction'].mean():.4f}")
        
        if 'confidence' in predictions_df.columns:
            print(f"\nConfiance moyenne: {predictions_df['confidence'].mean():.4f}")
else:
    print("\n✗ Erreur lors de l'exécution du pipeline. Vérifiez les logs ci-dessus.")

INFO:utils.prediction_pipeline:####################################################
INFO:utils.prediction_pipeline:### PIPELINE COMPLET D'INFÉRENCE EN PRODUCTION ###
INFO:utils.prediction_pipeline:### Date/Heure: 2026-04-02 10:55:37 ###
INFO:utils.prediction_pipeline:####################################################

INFO:utils.prediction_pipeline:=== ÉTAPE 1: CONFIGURATION ===
INFO:utils.inference_model:MLflow connecté à https://jenedai-mlflow.hf.space/
INFO:utils.prediction_pipeline:MLflow configuré
INFO:utils.database_handler:Moteur de base de données créé
INFO:utils.database_handler:Connexion à la base de données vérifiée
INFO:utils.database_handler:Tables créées avec succès
INFO:utils.prediction_pipeline:Base de données configurée
INFO:utils.prediction_pipeline:=== ÉTAPE 2: CHARGEMENT DU MODÈLE ===
INFO:botocore.credentials:Found credentials in environment variables.


INFO:utils.inference_model:Modèle enedis_model v4 chargé via alias 'prod'
INFO:utils.prediction_pipeline:Modèle chargé: {'model_type': 'RandomForestRegressor', 'version': '4', 'n_features': 36}
INFO:utils.prediction_pipeline:=== ÉTAPE 3: GÉNÉRATION DES DONNÉES ===
INFO:utils.data_generator:Données d'inférence générées: 72 échantillons sur 3 jours
INFO:utils.data_generator:Features: ['temperature', 'humidity', 'wind_speed', 'cloud_cover', 'day_of_week', 'hour', 'holiday']
INFO:utils.prediction_pipeline:=== ÉTAPE 4: EXÉCUTION DES PRÉDICTIONS ===
INFO:utils.data_generator:Features préparées: (72, 7)
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
ERROR:utils.inference_model:Erreur lors de la prédiction: X has 7 features, but RandomForestRegressor is expecting 36 features as input.
ERROR:utils.prediction_pipeline:Impossible de générer les prédictions



✗ Erreur lors de l'exécution du pipeline. Vérifiez les logs ci-dessus.


## Résultats